<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/svd/notebooks/01_1_svd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!test -d /content/recommendation-system/.git \
  || git clone -b svd https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin main svd
!git checkout main
!git reset --hard origin/main

# Create or reset and switch to the svd branch
!git checkout -B svd origin/svd

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds

from configs.model_configs import SVD_CONFIG
from src.data.build_matrix import add_domain_item_ids, build_explicit_svd_matrix
from src.evaluation.cross_domain_eval import evaluate_multi_domain


## Data preparation

In [ ]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [ ]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

In [ ]:
def fit_svd_recommender(train_interactions_df):
    interaction_matrix = build_explicit_svd_matrix(train_interactions_df)
    user_item_train = interaction_matrix.user_item_train

    row_counts = np.diff(user_item_train.indptr).clip(min=1)
    row_means = np.array(user_item_train.sum(axis=1)).flatten() / row_counts

    centered = user_item_train.copy().astype(np.float32)
    for user_idx in range(centered.shape[0]):
        start, end = centered.indptr[user_idx], centered.indptr[user_idx + 1]
        centered.data[start:end] -= row_means[user_idx]

    n_factors = min(SVD_CONFIG['n_factors'], min(centered.shape) - 1)
    if n_factors < 1:
        raise ValueError('SVD requires at least two users and two items')

    U, sigma, Vt = svds(centered, k=n_factors)
    U = U[:, ::-1]
    sigma = sigma[::-1]
    Vt = Vt[::-1, :]

    return {
        'interaction_matrix': interaction_matrix,
        'user_item_train': user_item_train,
        'user2idx': interaction_matrix.user2idx,
        'idx2item_array': np.array([
            interaction_matrix.idx2item[i]
            for i in range(len(interaction_matrix.idx2item))
        ]),
        'train_seen_idx_by_user': interaction_matrix.train_seen_idx_by_user,
        'domain_item_indices': interaction_matrix.domain_item_indices,
        'user_factors': U @ np.diag(sigma),
        'item_factors': Vt.T,
        'n_factors': n_factors,
    }


joint_svd = fit_svd_recommender(train_df)

user_item_train = joint_svd['user_item_train']
n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(f"n_factors={joint_svd['n_factors']}")


## Decomposing the matrix

In [ ]:
def make_svd_recommender(svd_state):
    user2idx = svd_state['user2idx']
    user_factors = svd_state['user_factors']
    item_factors = svd_state['item_factors']
    idx2item_array = svd_state['idx2item_array']
    train_seen_idx_by_user = svd_state['train_seen_idx_by_user']
    domain_item_indices = svd_state['domain_item_indices']

    def recommend_for_users(user_ids, target_domain, k=10, batch_size=None):
        if batch_size is None:
            batch_size = SVD_CONFIG['batch_size']

        domain_indices = domain_item_indices.get(target_domain, np.array([], dtype=np.int64))
        recommendations = {}
        user_ids = list(user_ids)

        if len(domain_indices) == 0:
            return {user_id: [] for user_id in user_ids}

        domain_item_factors = item_factors[domain_indices]
        domain_idx2item = idx2item_array[domain_indices]
        domain_position_by_idx = {
            item_idx: pos for pos, item_idx in enumerate(domain_indices)
        }

        for batch_start in range(0, len(user_ids), batch_size):
            batch_user_ids = user_ids[batch_start:batch_start + batch_size]
            valid_user_ids = []
            batch_user_idx = []

            for user_id in batch_user_ids:
                user_idx = user2idx.get(user_id)
                if user_idx is None:
                    recommendations[user_id] = []
                    continue
                valid_user_ids.append(user_id)
                batch_user_idx.append(user_idx)

            if not batch_user_idx:
                continue

            batch_scores = user_factors[np.array(batch_user_idx, dtype=np.int64)] @ domain_item_factors.T

            for row_idx, user_id in enumerate(valid_user_ids):
                user_idx = batch_user_idx[row_idx]
                scores = batch_scores[row_idx].copy()

                seen_positions = [
                    domain_position_by_idx[item_idx]
                    for item_idx in train_seen_idx_by_user.get(user_idx, set())
                    if item_idx in domain_position_by_idx
                ]
                if seen_positions:
                    scores[seen_positions] = -np.inf

                finite_count = np.isfinite(scores).sum()
                if finite_count == 0:
                    recommendations[user_id] = []
                    continue

                top_n = min(k, int(finite_count))
                top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
                top_positions = top_positions[np.argsort(-scores[top_positions])]
                recommendations[user_id] = list(domain_idx2item[top_positions])

        return recommendations

    return recommend_for_users


joint_recommend_for_users = make_svd_recommender(joint_svd)


In [ ]:
def svd_result_row(model_name, train_interactions_df, svd_state, valid_results, test_results):
    row = {
        'model': model_name,
        'n_factors': svd_state['n_factors'],
        'k': SVD_CONFIG['k'],
        'n_users': svd_state['user_item_train'].shape[0],
        'n_items': svd_state['user_item_train'].shape[1],
        'n_train_interactions': svd_state['user_item_train'].nnz,
    }

    for split_name, results in [('valid', valid_results), ('test', test_results)]:
        for metric in ['ndcg@10', 'recall@10', 'map@10']:
            values = []
            for domain in ['books', 'movies']:
                value = results.get(domain, {}).get(metric, np.nan)
                row[f'{domain}_{split_name}_{metric}'] = value
                if not pd.isna(value):
                    values.append(value)
            row[f'{split_name}_macro_{metric}'] = round(float(np.mean(values)), 4) if values else np.nan

    return row


In [ ]:
K = SVD_CONFIG['k']

joint_valid_results, joint_test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=joint_recommend_for_users,
    k=K,
)

joint_result_row = svd_result_row(
    'explicit_svd_books_movies_joint',
    train_df,
    joint_svd,
    joint_valid_results,
    joint_test_results,
)

joint_result_row


## Model evaluation


In [ ]:
books_only_svd = fit_svd_recommender(books_train)
books_only_recommend_for_users = make_svd_recommender(books_only_svd)

books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=books_only_recommend_for_users,
    k=K,
)

books_only_result_row = svd_result_row(
    'explicit_svd_books_only',
    books_train,
    books_only_svd,
    books_only_valid_results,
    books_only_test_results,
)

books_only_result_row


In [ ]:
movies_only_svd = fit_svd_recommender(movies_train)
movies_only_recommend_for_users = make_svd_recommender(movies_only_svd)

movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=movies_only_recommend_for_users,
    k=K,
)

movies_only_result_row = svd_result_row(
    'explicit_svd_movies_only',
    movies_train,
    movies_only_svd,
    movies_only_valid_results,
    movies_only_test_results,
)

movies_only_result_row


In [ ]:
svd_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    joint_result_row,
])

os.makedirs('results', exist_ok=True)
svd_ablation_results.to_csv('results/svd_ablation_metrics.csv', index=False)
svd_ablation_results
